# Prepare data for integration
We will test three different methods: scVI, sysVI, and scPli, as they allow us to extract batch embeddings

Set up kernel

In [1]:
# !bash -c 'eval "$(conda shell.bash hook)" \
#     && conda activate /work/islet_cartography_scrna/scrna_cartography_py_analysis \
#     && python -m ipykernel install --user \
#        --name scrna_cartography_py_analysis \
#        --display-name "py_analysis"'

Restart kernel - remember to manually select it in the top right corner

In [2]:
# import IPython
# app = IPython.get_ipython()
# app.kernel.do_shutdown(restart=True)

## Load libraries

In [3]:
import os
import sys
import glob
from pyhere import here

import pandas as pd
import anndata
import scanpy as sc
import seaborn as sns
from pathlib import Path
import scib
import numpy as np

# My modules / functions
sys.path.append(str(here('scripts/misc')))  
import my_anndata as ma

## Set up parameters

In [4]:
# Directories
ma.create_directories(dir_path = str(here('data/integration/first_pass')))
ma.create_directories(dir_path = str(here('data/integration/first_pass/tmp')))
ma.create_directories(dir_path = str(here('data/integration/first_pass/files')))
ma.create_directories(dir_path = str(here('data/integration/first_pass/plot')))
ma.create_directories(dir_path = str(here('data/integration/first_pass/objects')))
ma.create_directories(dir_path = str(here('data/integration/first_pass/models')))

/work/islet_cartography_scrna/data/integration/first_pass Directory already exists!
/work/islet_cartography_scrna/data/integration/first_pass/tmp Directory already exists!
/work/islet_cartography_scrna/data/integration/first_pass/files Directory already exists!
/work/islet_cartography_scrna/data/integration/first_pass/plot Directory already exists!
/work/islet_cartography_scrna/data/integration/first_pass/objects Directory already exists!
/work/islet_cartography_scrna/data/integration/first_pass/models Directory already exists!


In [5]:
# Paths 
adata_dir = str(here('data/anndata/'))
# Saving
save_dir = str(here('data/integration/first_pass/'))
plot_dir = str(here('data/integration/first_pass/plot/'))
tmp_dir = str(here('data/integration/first_pass/tmp/'))

In [6]:
# plotting
sc.set_figure_params(figsize=(5, 5))
# save path
sc.settings.figdir = plot_dir
%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"

/work/islet_cartography_scrna/scrna_cartography_py_analysis/lib/python3.13/site-packages/matplotlib_inline/config.py:54: DeprecationWarning: InlineBackend._print_figure_kwargs_changed is deprecated in traitlets 4.1: use @observe and @unobserve instead.
  def _update_figure_formatters(self):


## Load data

In [7]:
adata=sc.read_h5ad(os.path.join(adata_dir, 'AB_combined.h5ad'))

In [8]:
order = [
    "beta", "alpha", "alpha_beta", "delta", "gamma", "epsilon", "acinar", "ductal",
    "endothelial", "co_expression", "immune", "leukocyte", "mast", "mesenchymal",
    "mhc_class_ii", "schwann", "stellate", "endocrine", "cycling", "excluded"
]

## Add technical meta data

## Test which techniques to find hvgs by

In [9]:
# --- 0. Store full normalized+log data in .raw ---
adata.raw = adata.copy()

# --- 1. Define HVG cutoffs and technical covariates ---
hvg_cutoffs = [500, 1000, 2000]
tech_covariates = ["count_quantification", "library_prep", "cell_nuclei", "count_quantification|cell_nuclei"]

# --- 2. Define batch_keys to test ---
batch_keys_to_test = [
    None,  # global HVGs
    "count_quantification",
    "cell_nuclei",
    "library_prep",
    "count_quantification|cell_nuclei"  # small composite
]

# If composite keys are used, create them
composite_keys = ["count_quantification|cell_nuclei"]
for key in composite_keys:
    factors = key.split("|")
    present_factors = [f for f in factors if f in adata.obs.columns]
    if present_factors:
        adata.obs[key] = adata.obs['cell_nuclei'].astype(str) + "_" + adata.obs['count_quantification'].astype(str)

# Dictionary to store results
# Keys = (n_HVGs, batch_key_label)
hvg_results = {}
results_list = []

for n in hvg_cutoffs:
    print(f"\n=== HVG cutoff: {n} genes ===")
    for batch_key in batch_keys_to_test:
        key_label = batch_key if batch_key is not None else "global"
        print(f"\n--- Batch key: {key_label} ---")
        
        ad = adata.copy()
        
        # HVG detection
        sc.pp.highly_variable_genes(
            ad,
            n_top_genes=n,
            flavor="seurat_v3",
            layer="counts",
            batch_key=batch_key,
            subset=True
        )
        print(f"Selected {ad.n_vars} HVGs")
        
        # Store results
        hvg_results[(n, key_label)] = ad.var_names
        
        # --- PCA ---
        sc.tl.pca(ad, n_comps = 20, chunked=True, chunk_size=20000, mask_var="highly_variable")
        
        # --- PCA plots ---
        if batch_key is None:
            pp_order = order + list(np.unique(ad.obs[tech_covariates].values))
            color_keys = tech_covariates + ['study_cell_annotation_harmonized']
            sc.pl.pca_scatter(ad, size = 2, color= color_keys, ncols = len(color_keys), groups = pp_order, wspace=1, show=False,  save=os.path.join(f"_PCA_{n}_HVGs_global.png"))
        else:          
            # define order to plot
            pp_order = order + list(np.unique(ad.obs[key_label].values))
            sc.pl.pca_scatter(ad, size = 2, color=[key_label, 'study_cell_annotation_harmonized'], groups = pp_order, wspace=1, ncols=2, show=False, save=os.path.join(f"_PCA_{n}_HVGs_{key_label}.png"))

        # save adata
        ad.write_h5ad(os.path.join(tmp_dir, f'adata_{n}HVGs_{key_label}.h5ad'))
        
        # --- PC regression for global ---
       
        # Only do PC regression for global HVGs
        if batch_key is None:
            pcs = ad.obsm['X_pca']
            for cov in tech_covariates:
                if cov in ad.obs.columns:
                    cov_series = ad.obs[cov]
                    r2_mean = scib.metrics.pc_regression(
                        data=pcs,
                        n_comps=20,
                        covariate=cov_series,
                        n_threads=60,
                        verbose=False
                    )
                    
                    print(f"  Covariate {cov}, mean R² = {r2_mean:.3f}")
            
                    results_list.append({
                        "n_HVGs": n,
                        "batch_key": key_label,
                        "covariate": cov,
                        "r2_mean": r2_mean
                    })

pc_regression_df = pd.DataFrame(results_list)

# --- 3. HVG overlap summary ---
for n in hvg_cutoffs:
    print(f"\n--- HVG overlap summary for {n} genes ---")
    # Compare each batch_key with global
    hvgs_global = set(hvg_results[(n, "global")])
    for batch_key in batch_keys_to_test:
        key_label = batch_key if batch_key is not None else "global"
        if key_label == "global":
            continue
        hvgs_batch = set(hvg_results[(n, key_label)])
        overlap = len(hvgs_global.intersection(hvgs_batch))
        print(f"Batch key {key_label}: {overlap} genes shared with global HVGs")


=== HVG cutoff: 500 genes ===

--- Batch key: global ---
Selected 500 HVGs


/work/islet_cartography_scrna/scrna_cartography_py_analysis/lib/python3.13/site-packages/scanpy/preprocessing/_pca/__init__.py:227: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(


  Covariate count_quantification, mean R² = 0.002


/work/islet_cartography_scrna/scrna_cartography_py_analysis/lib/python3.13/site-packages/scanpy/preprocessing/_pca/__init__.py:227: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(


  Covariate library_prep, mean R² = 0.054


...

  Covariate library_prep, mean R² = 0.099


/work/islet_cartography_scrna/scrna_cartography_py_analysis/lib/python3.13/site-packages/scanpy/preprocessing/_pca/__init__.py:227: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(


  Covariate cell_nuclei, mean R² = 0.066


/work/islet_cartography_scrna/scrna_cartography_py_analysis/lib/python3.13/site-packages/scanpy/preprocessing/_pca/__init__.py:227: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(


  Covariate count_quantification|cell_nuclei, mean R² = 0.069

--- Batch key: count_quantification ---
Selected 2000 HVGs

--- Batch key: cell_nuclei ---
Selected 2000 HVGs

--- Batch key: library_prep ---
Selected 2000 HVGs

--- Batch key: count_quantification|cell_nuclei ---
Selected 2000 HVGs

--- HVG overlap summary for 500 genes ---
Batch key count_quantification: 288 genes shared with global HVGs
Batch key cell_nuclei: 268 genes shared with global HVGs
Batch key library_prep: 132 genes shared with global HVGs
Batch key count_quantification|cell_nuclei: 217 genes shared with global HVGs

--- HVG overlap summary for 1000 genes ---
Batch key count_quantification: 482 genes shared with global HVGs
Batch key cell_nuclei: 525 genes shared with global HVGs
Batch key library_prep: 281 genes shared with global HVGs
Batch key count_quantification|cell_nuclei: 381 genes shared with global HVGs

--- HVG overlap summary for 2000 genes ---
Batch key count_quantification: 935 genes shared with 

In [18]:
pc_regression_df.to_csv(os.path.join(here('data/integration/first_pass/files'), "pc_regression_global_hvgs.csv"), index=True)

In [19]:
rows = []
for n in hvg_cutoffs:
    hvgs_global = set(hvg_results[(n, "global")])
    for batch_key in batch_keys_to_test:
        key_label = batch_key if batch_key is not None else "global"
        if key_label == "global":
            continue
        hvgs_batch = set(hvg_results[(n, key_label)])
        overlap = len(hvgs_global.intersection(hvgs_batch))
        rows.append({
            "n_genes": n,
            "batch_key": key_label,
            "overlap_with_global": overlap
        })

overlap_df = pd.DataFrame(rows)

In [33]:
rows = []
for (n, batch_key), gene_index in hvg_results.items():
    # normalize a None batch_key if needed (your dataset uses 'global' as a string already)
    key_label = batch_key if batch_key is not None else "global"
    # gene_index is a pandas.Index; enumerate to keep rank
    for rank, gene in enumerate(gene_index, start=1):
        rows.append({
            "n_genes": int(n),
            "batch_key": str(key_label),
            "gene_symbol": str(gene)
        })

hvg_long = pd.DataFrame(rows)
# optional: tidy dtypes
hvg_long = hvg_long.astype({"n_genes": "int64", "batch_key": "string", "gene_symbol": "string"})
hvg_long.head()
hvg_long.to_csv(os.path.join(here('data/integration/first_pass/files'), "hvgs.csv"), index=True)

In [10]:
# import scanpy as sc
# import pandas as pd
# from scipy.stats import chi2_contingency

# # Assuming `adata` is your AnnData object and HVGs are already computed
# # HVGs are typically stored in: adata.var['highly_variable']

# # Check how many HVGs are associated with each batch
# # First, make sure you have a batch column in `adata.obs`, e.g., 'batch'
# # Then, map each gene to the batches where it's expressed

# # Create a DataFrame with gene expression presence per batch
# gene_presence = pd.DataFrame({
#     'gene': adata.var_names,
#     'is_hvg': adata.var['highly_variable']
# })

# # Count how many HVGs are expressed in each batch
# batch_counts = adata.obs['batch'].value_counts()
# hvg_counts_by_batch = adata[adata.var['highly_variable']].obs['batch'].value_counts()

# # Chi-squared test to check for skew
# observed = hvg_counts_by_batch.values
# expected = batch_counts.values * (sum(hvg_counts_by_batch) / sum(batch_counts))
# chi2, p, _, _ = chi2_contingency([observed, expected])

# print(f"Chi-squared test p-value: {p}")
# if p < 0.05:
#     print("There is a statistically significant skew in HVG selection across batches.")
# else:
#     print("No significant skew in HVG selection across batches.")


KeyError: 'highly_variable'